# Repair corporate-action discontinuities in `MatrixStoreMinuteFeed` data

This notebook scans your **cube / matrix store** (`date=YYYY-MM-DD/<field>.parquet`) for **split / bonus-like overnight price discontinuities**, reviews the detected events, and writes a **repaired store** that you can point `MatrixStoreMinuteFeed` at.

The goal is to make the minute-level price series **continuous on a post-event basis**, so synthetic portfolio / execution logic does not collapse when a stock suddenly appears to fall by 70–80% because the historical data was not adjusted.

## What this notebook does

1. Reads daily `close.parquet` files from the cube store.
2. Compares the **previous day’s robust last price** with the **current day’s robust first price** for every symbol.
3. Flags suspicious overnight jumps that look like **splits / bonus issues**.
4. Snaps raw ratios to plausible corporate-action factors (for example `2.0`, `3.0`, `1.5`, `1.25`, etc.).
5. Builds cumulative backward adjustment factors.
6. Rewrites the store into a new output directory.
7. Saves an audit trail:
   - `detected_split_events.csv`
   - `repair_manifest.json`

## Important safety note

By default this notebook writes to a **new output directory**. That is safer than editing the live cube store in place.

If you rebuild the cube store from raw CSVs later, you will need to either:
- rerun this notebook, or
- fix the raw per-symbol source files upstream and rebuild.

In [1]:
from __future__ import annotations

import json
import shutil
import sys
from pathlib import Path
from typing import Dict, Iterable, List, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Configuration

Change the paths below to match your machine.

A few practical defaults are already chosen for your use case:

- `MIN_RAW_FACTOR = 1.8`  
  This targets the kind of **70–80% “fake crash”** you mentioned and avoids over-flagging normal market gaps.
- `REL_TOL = 0.08`  
  The raw overnight ratio must be reasonably close to a plausible split / bonus factor.
- `WINDOW_MINUTES = 15`  
  Uses a median over multiple minutes instead of a single noisy tick.

In [11]:
# --- project / repo paths ---
REPO_ROOT = Path(r"E:/trading_simulator/trading_sim_v2")   # change if needed
if REPO_ROOT.exists() and str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Optional: used only in the validation cell near the end
try:
    from market_feed.matrix_store_1m import MatrixStoreMinuteFeed
except Exception:
    MatrixStoreMinuteFeed = None

# --- matrix / cube store paths ---
STORE_DIR = Path(r"E:/data_1m/processed_data/nifty500/1m_cube_store_repaired")
OUTPUT_STORE_DIR = Path(r"E:/data_1m/processed_data/nifty500/1m_cube_store_repaired_v2")

# --- field to use for detection ---
DETECT_FIELD = "close"

# --- fields to adjust if they exist in each date partition ---
PRICE_FIELDS = ["open", "high", "low", "close"]
VOLUME_FIELDS = ["volume"]

# --- detection parameters ---
MIN_RAW_FACTOR = 1.8
MAX_RAW_FACTOR = 20.0
REL_TOL = 0.08
WINDOW_MINUTES = 15
MAX_SPLIT_FACTOR = 20
MAX_BONUS_NUM = 10
MAX_BONUS_DEN = 10

# --- output behavior ---
OVERWRITE_OUTPUT = False
COPY_UNADJUSTED_FILES = True

# --- audit files ---
EVENTS_CSV = OUTPUT_STORE_DIR.parent / "detected_split_events.csv"
MANIFEST_JSON = OUTPUT_STORE_DIR.parent / "repair_manifest.json"

print("STORE_DIR       =", STORE_DIR)
print("OUTPUT_STORE_DIR=", OUTPUT_STORE_DIR)

STORE_DIR       = E:\data_1m\processed_data\nifty500\1m_cube_store_repaired
OUTPUT_STORE_DIR= E:\data_1m\processed_data\nifty500\1m_cube_store_repaired_v2


## Helper functions

In [3]:
def build_candidate_factors(
    max_split: int = 20,
    max_bonus_num: int = 10,
    max_bonus_den: int = 10,
) -> List[float]:
    """
    Build a list of plausible corporate-action factors (> 1.0).

    Examples:
    - 2-for-1 split  -> factor 2.0
    - 5-for-1 split  -> factor 5.0
    - 1:1 bonus      -> factor 2.0
    - 1:2 bonus      -> factor 1.5
    - 3:5 bonus      -> factor 1.6
    """
    factors = set()

    for n in range(2, max_split + 1):
        factors.add(float(n))

    for a in range(1, max_bonus_num + 1):
        for b in range(1, max_bonus_den + 1):
            factors.add((a + b) / b)

    return sorted(f for f in factors if f > 1.0)


def snap_factor(
    raw_factor: float,
    candidate_factors: Iterable[float],
    rel_tol: float = 0.08,
) -> tuple[float | None, float | None]:
    """
    Snap the raw overnight ratio to the nearest plausible corporate-action factor.
    Returns:
      (chosen_factor, relative_error)
    """
    if not np.isfinite(raw_factor) or raw_factor <= 1.0:
        return None, None

    candidates = np.array(list(candidate_factors), dtype=float)
    idx = int(np.argmin(np.abs(candidates - raw_factor)))
    chosen = float(candidates[idx])
    rel_err = abs(chosen - raw_factor) / chosen

    if rel_err <= rel_tol:
        return chosen, rel_err

    return None, rel_err


def list_date_dirs(store_dir: Path) -> List[Path]:
    return sorted([p for p in store_dir.glob("date=*") if p.is_dir()])


def load_field_day(path: Path) -> pd.DataFrame:
    df = pd.read_parquet(path)
    if "ts" not in df.columns:
        raise ValueError(f"'ts' column missing in {path}")
    df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
    return df.sort_values("ts").reset_index(drop=True)


def numeric_body(df: pd.DataFrame) -> pd.DataFrame:
    body = df.drop(columns=["ts"], errors="ignore").copy()
    for c in body.columns:
        body[c] = pd.to_numeric(body[c], errors="coerce")
    return body


def robust_first_reference(body: pd.DataFrame, n: int = 15) -> pd.Series:
    """Per symbol, take the median of the first N valid values of the day."""
    def fn(col: pd.Series) -> float:
        vals = col.dropna().iloc[:n]
        return float(vals.median()) if len(vals) else np.nan
    return body.apply(fn, axis=0)


def robust_last_reference(body: pd.DataFrame, n: int = 15) -> pd.Series:
    """Per symbol, take the median of the last N valid values of the day."""
    def fn(col: pd.Series) -> float:
        vals = col.dropna().iloc[-n:]
        return float(vals.median()) if len(vals) else np.nan
    return body.apply(fn, axis=0)

In [13]:
def detect_split_events(
    store_dir: Path,
    detect_field: str = "close",
    min_raw_factor: float = 1.8,
    max_raw_factor: float = 20.0,
    rel_tol: float = 0.08,
    window_minutes: int = 15,
    candidate_factors: Optional[List[float]] = None,
) -> pd.DataFrame:
    """
    Detect split / bonus-like discontinuities from the cube store.

    Method:
    - previous day's robust last reference price
    - current day's robust first reference price
    - raw_factor = prev_ref / curr_ref

    If raw_factor is large and close to a plausible corporate-action factor,
    we flag it.

    This is intentionally focused on the user's failure mode:
    unadjusted downward price jumps that break portfolio logic.
    """
    if candidate_factors is None:
        candidate_factors = build_candidate_factors(
            max_split=MAX_SPLIT_FACTOR,
            max_bonus_num=MAX_BONUS_NUM,
            max_bonus_den=MAX_BONUS_DEN,
        )

    date_dirs = list_date_dirs(store_dir)
    if not date_dirs:
        raise FileNotFoundError(f"No date=* partitions found under {store_dir}")

    records = []
    prev_date = None
    prev_last = None

    for dd in date_dirs:
        date_str = dd.name.split("=", 1)[1]
        field_path = dd / f"{detect_field}.parquet"
        if not field_path.exists():
            continue

        df = load_field_day(field_path)
        body = numeric_body(df)

        curr_first = robust_first_reference(body, n=window_minutes)
        curr_last = robust_last_reference(body, n=window_minutes)

        if prev_last is not None:
            common = prev_last.index.intersection(curr_first.index)
            if len(common):
                raw_ratios = (prev_last[common] / curr_first[common]).replace([np.inf, -np.inf], np.nan).dropna()
                raw_ratios = raw_ratios[(raw_ratios >= min_raw_factor) & (raw_ratios <= max_raw_factor)]

                for sym, raw_factor in raw_ratios.items():
                    snapped, rel_err = snap_factor(float(raw_factor), candidate_factors, rel_tol=rel_tol)
                    if snapped is None:
                        continue

                    records.append(
                        {
                            "symbol": sym,
                            "prev_date": prev_date,
                            "event_date": date_str,
                            "prev_ref": float(prev_last[sym]),
                            "curr_ref": float(curr_first[sym]),
                            "event_day_last_ref": float(curr_last.get(sym, np.nan)),
                            "raw_factor": float(raw_factor),
                            "snapped_factor": float(snapped),
                            "snap_rel_error": float(rel_err if rel_err is not None else np.nan),
                            "implied_drop_pct": float(1.0 - 1.0 / raw_factor),
                        }
                    )

        prev_date = date_str
        prev_last = curr_last

    out = pd.DataFrame(records)
    if out.empty:
        return out

    out = out.sort_values(["event_date", "symbol"]).reset_index(drop=True)
    return out


def build_daily_adjustment_table(date_index: List[str], events: pd.DataFrame) -> pd.DataFrame:
    """
    For each date and symbol, build the cumulative backward adjustment factor.

    Interpretation:
    - factor on date D = 1.0 means no change on that date
    - factor > 1.0 means prices on that date should be divided by factor
      so that history is expressed on the post-event basis
    """
    if events.empty:
        return pd.DataFrame(index=pd.Index(date_index, name="date"))

    symbols = sorted(events["symbol"].unique())
    adj = pd.DataFrame(1.0, index=pd.Index(date_index, name="date"), columns=symbols, dtype="float64")

    event_map: Dict[str, Dict[str, float]] = {}
    for _, row in events.iterrows():
        event_map.setdefault(row["event_date"], {})
        event_map[row["event_date"]][row["symbol"]] = (
            event_map[row["event_date"]].get(row["symbol"], 1.0) * float(row["snapped_factor"])
        )

    running = pd.Series(1.0, index=adj.columns, dtype="float64")

    for d in reversed(date_index):
        adj.loc[d] = running
        if d in event_map:
            for sym, factor in event_map[d].items():
                running.loc[sym] *= factor

    return adj


def repair_matrix_store(
    input_store_dir: Path,
    output_store_dir: Path,
    events: pd.DataFrame,
    price_fields: Optional[List[str]] = None,
    volume_fields: Optional[List[str]] = None,
    copy_unadjusted_files: bool = True,
    overwrite: bool = False,
) -> Dict[str, object]:
    """
    Write a repaired matrix store.

    Price fields:
      divide older history by the cumulative adjustment factor

    Volume fields:
      multiply older history by the cumulative adjustment factor
    """
    price_fields = price_fields or ["open", "high", "low", "close"]
    volume_fields = volume_fields or ["volume"]

    date_dirs = list_date_dirs(input_store_dir)
    date_index = [dd.name.split("=", 1)[1] for dd in date_dirs]
    adj_table = build_daily_adjustment_table(date_index, events)

    if output_store_dir.exists():
        if overwrite:
            shutil.rmtree(output_store_dir)
        else:
            raise FileExistsError(
                f"Output directory already exists: {output_store_dir}\n"
                "Set OVERWRITE_OUTPUT = True if you want to replace it."
            )

    output_store_dir.mkdir(parents=True, exist_ok=True)

    files_written = 0
    files_adjusted = 0

    for dd in date_dirs:
        date_str = dd.name.split("=", 1)[1]
        out_dd = output_store_dir / dd.name
        out_dd.mkdir(parents=True, exist_ok=True)

        adj_row = adj_table.loc[date_str] if (not adj_table.empty and date_str in adj_table.index) else pd.Series(dtype="float64")

        for src in sorted(dd.iterdir()):
            if not src.is_file():
                continue

            dst = out_dd / src.name

            if src.suffix != ".parquet":
                if copy_unadjusted_files:
                    shutil.copy2(src, dst)
                    files_written += 1
                continue

            field = src.stem
            is_price_field = field in price_fields
            is_volume_field = field in volume_fields

            if not (is_price_field or is_volume_field):
                if copy_unadjusted_files:
                    shutil.copy2(src, dst)
                    files_written += 1
                continue

            df = load_field_day(src)
            cols = [c for c in df.columns if c != "ts"]

            if not cols:
                df.to_parquet(dst, index=False)
                files_written += 1
                continue

            effective = adj_row.reindex(cols).fillna(1.0) if len(adj_row) else pd.Series(1.0, index=cols)

            body = numeric_body(df)

            if is_price_field:
                body = body.divide(effective, axis=1)
            elif is_volume_field:
                body = body.multiply(effective, axis=1)

            out = pd.concat([df[["ts"]], body], axis=1)
            out.to_parquet(dst, index=False)

            files_written += 1
            files_adjusted += 1

    summary = {
        "input_store_dir": str(input_store_dir),
        "output_store_dir": str(output_store_dir),
        "dates_processed": len(date_dirs),
        "events_detected": int(len(events)),
        "symbols_flagged": int(events["symbol"].nunique()) if not events.empty else 0,
        "files_written": int(files_written),
        "files_adjusted": int(files_adjusted),
        "price_fields": list(price_fields),
        "volume_fields": list(volume_fields),
    }
    return summary

## Run detection

In [14]:
candidate_factors = build_candidate_factors(
    max_split=MAX_SPLIT_FACTOR,
    max_bonus_num=MAX_BONUS_NUM,
    max_bonus_den=MAX_BONUS_DEN,
)

events = detect_split_events(
    store_dir=STORE_DIR,
    detect_field=DETECT_FIELD,
    min_raw_factor=MIN_RAW_FACTOR,
    max_raw_factor=MAX_RAW_FACTOR,
    rel_tol=REL_TOL,
    window_minutes=WINDOW_MINUTES,
    candidate_factors=candidate_factors,
)

print(f"Detected events: {len(events)}")
print(f"Symbols flagged: {events['symbol'].nunique() if not events.empty else 0}")
events.head(20)

Detected events: 0
Symbols flagged: 0


""


## Review the flagged events before writing anything

Start here. This is the most important review step.

Things to look for:
- repeated events on the same symbol across time
- factors like `2.0`, `3.0`, `5.0`, `10.0`
- dates that match known split / bonus issue windows

In [15]:
if events.empty:
    print("No events detected.")
else:
    display(
        events.sort_values(["symbol", "event_date"])
              .reset_index(drop=True)
              .head(200)
    )

No events detected.


In [7]:
if not events.empty:
    summary_by_symbol = (
        events.groupby("symbol")
        .agg(
            n_events=("symbol", "size"),
            first_event=("event_date", "min"),
            last_event=("event_date", "max"),
            factors=("snapped_factor", lambda s: sorted(set(np.round(s.astype(float), 6))))
        )
        .sort_values(["n_events", "last_event"], ascending=[False, False])
        .reset_index()
    )
    display(summary_by_symbol.head(100))
else:
    summary_by_symbol = pd.DataFrame()
    print("No symbol summary because no events were detected.")

,symbol,n_events,first_event,last_event,factors
0,PIIND,2,2018-12-03,2019-01-03,[9.0]
1,GUJGASLTD,2,2015-10-01,2018-12-26,[5.0]
2,ABFRL,1,2025-05-22,2025-05-22,[1.888889]
3,TATACHEM,1,2020-03-04,2020-03-04,[2.111111]
4,CGPOWER,1,2016-03-15,2016-03-15,[3.0]
5,JSL,1,2015-11-19,2015-11-19,[2.666667]
6,INDIAMART,1,2015-10-21,2015-10-21,[1.9]
7,ZYDUSLIFE,1,2015-10-06,2015-10-06,[5.0]
8,COLPAL,1,2015-09-23,2015-09-23,[2.0]
9,DIVISLAB,1,2015-09-23,2015-09-23,[2.0]


## Optional: inspect one flagged event visually

Set `SYMBOL_TO_PLOT` and `EVENT_DATE_TO_PLOT` from the detection table.

In [ ]:
def plot_event_window(store_dir: Path, symbol: str, event_date: str, detect_field: str = "close"):
    dates = [p.name.split("=", 1)[1] for p in list_date_dirs(store_dir)]
    if event_date not in dates:
        raise ValueError(f"event_date {event_date} not found in store")

    idx = dates.index(event_date)
    if idx == 0:
        raise ValueError("Cannot plot previous day because event_date is the first available date")

    prev_date = dates[idx - 1]

    prev_path = store_dir / f"date={prev_date}" / f"{detect_field}.parquet"
    curr_path = store_dir / f"date={event_date}" / f"{detect_field}.parquet"

    prev_df = load_field_day(prev_path)
    curr_df = load_field_day(curr_path)

    if symbol not in prev_df.columns or symbol not in curr_df.columns:
        raise KeyError(f"{symbol} not found in both days")

    prev_s = pd.to_numeric(prev_df[symbol], errors="coerce")
    curr_s = pd.to_numeric(curr_df[symbol], errors="coerce")

    fig = plt.figure(figsize=(12, 4))
    plt.plot(prev_df["ts"], prev_s, label=f"{symbol} | {prev_date}")
    plt.plot(curr_df["ts"], curr_s, label=f"{symbol} | {event_date}")
    plt.title(f"{symbol}: raw prices around suspected event")
    plt.xlabel("Timestamp")
    plt.ylabel("Price")
    plt.legend()
    plt.grid(True)
    plt.show()

# Example:
# SYMBOL_TO_PLOT = events.iloc[0]["symbol"]
# EVENT_DATE_TO_PLOT = events.iloc[0]["event_date"]
# plot_event_window(STORE_DIR, SYMBOL_TO_PLOT, EVENT_DATE_TO_PLOT, detect_field=DETECT_FIELD)

## Save the detection audit

In [ ]:
OUTPUT_STORE_DIR.parent.mkdir(parents=True, exist_ok=True)

if events.empty:
    print("No audit file written because no events were detected.")
else:
    events.to_csv(EVENTS_CSV, index=False)
    print("Wrote:", EVENTS_CSV)

## Build the repaired matrix store

This cell writes the repaired store to `OUTPUT_STORE_DIR`.

It does **not** overwrite the input store unless you deliberately point both paths to the same location.

In [8]:
if events.empty:
    repair_summary = {
        "input_store_dir": str(STORE_DIR),
        "output_store_dir": str(OUTPUT_STORE_DIR),
        "events_detected": 0,
        "status": "nothing_to_repair",
    }
else:
    repair_summary = repair_matrix_store(
        input_store_dir=STORE_DIR,
        output_store_dir=OUTPUT_STORE_DIR,
        events=events,
        price_fields=PRICE_FIELDS,
        volume_fields=VOLUME_FIELDS,
        copy_unadjusted_files=COPY_UNADJUSTED_FILES,
        overwrite=OVERWRITE_OUTPUT,
    )

repair_summary

{'input_store_dir': 'E:\\data_1m\\processed_data\\nifty500\\1m_cube_store',
 'output_store_dir': 'E:\\data_1m\\processed_data\\nifty500\\1m_cube_store_repaired',
 'dates_processed': 2587,
 'events_detected': 22,
 'symbols_flagged': 20,
 'files_written': 12935,
 'files_adjusted': 12935,
 'price_fields': ['open', 'high', 'low', 'close'],
 'volume_fields': ['volume']}

## Save manifest

In [ ]:
manifest = {
    "config": {
        "STORE_DIR": str(STORE_DIR),
        "OUTPUT_STORE_DIR": str(OUTPUT_STORE_DIR),
        "DETECT_FIELD": DETECT_FIELD,
        "PRICE_FIELDS": PRICE_FIELDS,
        "VOLUME_FIELDS": VOLUME_FIELDS,
        "MIN_RAW_FACTOR": MIN_RAW_FACTOR,
        "MAX_RAW_FACTOR": MAX_RAW_FACTOR,
        "REL_TOL": REL_TOL,
        "WINDOW_MINUTES": WINDOW_MINUTES,
        "MAX_SPLIT_FACTOR": MAX_SPLIT_FACTOR,
        "MAX_BONUS_NUM": MAX_BONUS_NUM,
        "MAX_BONUS_DEN": MAX_BONUS_DEN,
    },
    "summary": repair_summary,
}

with open(MANIFEST_JSON, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Wrote:", MANIFEST_JSON)

## Quick validation: compare raw vs repaired for one event

In [9]:
def compare_raw_vs_repaired(store_dir_raw: Path, store_dir_fixed: Path, symbol: str, event_date: str, field: str = "close"):
    dates = [p.name.split("=", 1)[1] for p in list_date_dirs(store_dir_raw)]
    if event_date not in dates:
        raise ValueError(f"{event_date} not found")

    idx = dates.index(event_date)
    if idx == 0:
        raise ValueError("Need a previous day for comparison")

    prev_date = dates[idx - 1]

    raw_prev = load_field_day(store_dir_raw / f"date={prev_date}" / f"{field}.parquet")
    raw_curr = load_field_day(store_dir_raw / f"date={event_date}" / f"{field}.parquet")

    fix_prev = load_field_day(store_dir_fixed / f"date={prev_date}" / f"{field}.parquet")
    fix_curr = load_field_day(store_dir_fixed / f"date={event_date}" / f"{field}.parquet")

    fig = plt.figure(figsize=(12, 4))
    plt.plot(raw_prev["ts"], pd.to_numeric(raw_prev[symbol], errors="coerce"), label=f"RAW {prev_date}")
    plt.plot(raw_curr["ts"], pd.to_numeric(raw_curr[symbol], errors="coerce"), label=f"RAW {event_date}")
    plt.plot(fix_prev["ts"], pd.to_numeric(fix_prev[symbol], errors="coerce"), linestyle="--", label=f"FIXED {prev_date}")
    plt.plot(fix_curr["ts"], pd.to_numeric(fix_curr[symbol], errors="coerce"), linestyle="--", label=f"FIXED {event_date}")
    plt.title(f"{symbol}: raw vs repaired")
    plt.xlabel("Timestamp")
    plt.ylabel(field)
    plt.legend()
    plt.grid(True)
    plt.show()

# Example:
# if not events.empty:
#     sym = events.iloc[0]["symbol"]
#     ed  = events.iloc[0]["event_date"]
#     compare_raw_vs_repaired(STORE_DIR, OUTPUT_STORE_DIR, sym, ed, field=DETECT_FIELD)

## Use the repaired store with `MatrixStoreMinuteFeed`

After validation, point the feed to `OUTPUT_STORE_DIR`.

In [10]:
if MatrixStoreMinuteFeed is None:
    print("MatrixStoreMinuteFeed import not available in this notebook session.")
else:
    TEST_SYMBOL = None
    TEST_START = None
    TEST_END = None

    if TEST_SYMBOL and TEST_START and TEST_END:
        feed = MatrixStoreMinuteFeed(
            store_dir=str(OUTPUT_STORE_DIR),
            start=pd.Timestamp(TEST_START).to_pydatetime(),
            end=pd.Timestamp(TEST_END).to_pydatetime(),
            symbols=[TEST_SYMBOL],
            speed="fast",
        )
        print(feed)
    else:
        print(
            "Set TEST_SYMBOL / TEST_START / TEST_END if you want to instantiate the feed "
            "against the repaired store."
        )

Set TEST_SYMBOL / TEST_START / TEST_END if you want to instantiate the feed against the repaired store.


## Notes on threshold tuning

If you later want to catch smaller corporate actions too, you can lower `MIN_RAW_FACTOR`, for example:

- `1.8`  -> safer, targets the kind of 70–80% discontinuity you described
- `1.5`  -> catches more bonus issues, but needs more manual review
- `1.3`  -> aggressive, higher false-positive risk

For a production repair workflow, the safest pattern is:

1. Run detection
2. Review `events`
3. Keep the good events
4. Write the repaired store

If needed, you can manually filter `events` before the repair cell. Example:

```python
events = events[~((events["symbol"] == "XYZ") & (events["event_date"] == "2020-03-24"))]
```